# Rakuten colours (ENS Challenge Data #59): 5-fold base models, stacking and catalogue neighbours

Full pipeline on Colab, with the data on Google Drive:
1. **image**: frozen SigLIP so400m embeddings + zero-shot colour scores, then a multi-label MLP head;
2. **text**: character TF-IDF + one logistic regression per colour, and a fine-tuned `tohoku-nlp/bert-base-japanese-v3`;
3. every model is trained **in 5 folds**: out-of-fold probabilities on the **212,120** training products, and the mean
   of the 5 fold models on the test set;
4. **stacking** trained on these 212,120 rows; then the **catalogue neighbours** (same shop, close item numbers) and
   the shop's usual colours (out-of-fold) are added, with one XGBoost meta-model per colour;
5. one threshold per colour, at least one colour per product; the better of the two stackers is submitted.

Step 4 also exists as a local script, `stack_neighbours.py` (LightGBM): it produced the final submission.

### Before running
In Google Drive, a folder **`rakuten_couleurs/`** (name set in `DRIVE_DIR` below) holding `X_train_12tkObq.csv`,
`X_test_gDTIJPh.csv`, `y_train_Q9n2dCu.csv` and the image archive `images_NFkV58V.tar` (challenge files).
An **A100** GPU is recommended.

### Resuming after a disconnection
The image embeddings (`out/siglip_feats.npz`) and every fold of every model (`out/oof5/<model>_f<k>.npz`) are saved
as soon as they are done, and skipped if they already exist: after a disconnection, run all cells again.

### Measured durations (A100, 2026-10-24)
SigLIP embeddings ≈ 35 min (measured on an L4); image head ≈ 1 min; TF-IDF ≈ 10 min per fold; BERT ≈ 23 min per fold;
stacking + neighbours a few minutes on GPU.

⚠ **Confidentiality**: the challenge data is confidential; Drive and Colab are Google services.

In [ ]:
!pip -q install open_clip_torch fugashi unidic-lite "transformers>=4.45,<4.47" "accelerate>=0.34" datasets
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, pathlib
DRIVE_DIR = "rakuten_couleurs"          # Drive folder holding the challenge files
D = pathlib.Path("/content/drive/MyDrive") / DRIVE_DIR
OUT = D / "out"
OOFD = OUT / "oof5"; OOFD.mkdir(parents=True, exist_ok=True)
print(sorted(p.name for p in OOFD.iterdir()))

In [ ]:
# ---- Settings
SEED = 42
N_FOLDS = 5
MODELS = ["image", "tfidf", "bert"]   # folds already in out/oof5/ are skipped
TEXT_MODEL = "tohoku-nlp/bert-base-japanese-v3"
TEXT_EPOCHS = 3
TEXT_MAX_LEN = 256
TEXT_BS = 64
TEXT_LR = 5e-5
BERT_NAME = f"bert_e{TEXT_EPOCHS}_l{TEXT_MAX_LEN}"

In [ ]:
import ast, re, html, gc
import numpy as np, pandas as pd, torch
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score
from sklearn.model_selection import KFold

Xtr = pd.read_csv(D / "X_train_12tkObq.csv", index_col=0)
Xte = pd.read_csv(D / "X_test_gDTIJPh.csv", index_col=0)
ytr = pd.read_csv(D / "y_train_Q9n2dCu.csv", index_col=0)["color_tags"].map(ast.literal_eval)
mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(ytr).astype(np.float32)
COLORS = list(mlb.classes_)
print(len(Xtr), len(Xte), len(COLORS), "colours")

def clean(s):
    if not isinstance(s, str):
        return ""
    s = re.sub(r"<[^>]+>", " ", html.unescape(s))
    return re.sub(r"\s+", " ", s).strip()

for X in (Xtr, Xte):
    X["text"] = (X["item_name"].map(clean) + " 。 " + X["item_caption"].map(clean)).str.strip(" 。")

# Fixed folds, shared by every model (do not change SEED between runs)
FOLDS = list(KFold(N_FOLDS, shuffle=True, random_state=SEED).split(np.zeros(len(Xtr))))
dev = "cuda" if torch.cuda.is_available() else "cpu"
amp = torch.bfloat16 if dev == "cuda" and torch.cuda.is_bf16_supported() else torch.float16

def wf1(y_true, y_pred):
    return f1_score(y_true, y_pred, average="weighted", zero_division=0)

def fold_file(name, k):
    return OOFD / f"{name}_f{k}.npz"

def todo(name):
    return [k for k in range(N_FOLDS) if not fold_file(name, k).exists()]

def merge(name):
    """Out-of-fold probabilities on the whole training set and mean of the 5 folds on the test set; None if a fold is missing."""
    if todo(name):
        return None
    oof, test = np.zeros(Y.shape, np.float32), np.zeros((len(Xte), Y.shape[1]), np.float32)
    for k, (_, b) in enumerate(FOLDS):
        z = np.load(fold_file(name, k))
        oof[b] = z["val"]; test += z["test"] / N_FOLDS
    return oof, test

GRID = np.arange(0.05, 0.96, 0.01)

def decide(p, th):
    pred = (p > th).astype(int)
    empty = pred.sum(1) == 0                       # never an empty list
    pred[empty, p[empty].argmax(1)] = 1
    return pred

def best_thresholds(p, y, chunk=20000):
    # one threshold per colour (the weighted F1 is a mean of per-colour F1 scores); computed in chunks (212k rows)
    tp = np.zeros((len(GRID), y.shape[1])); npred = np.zeros_like(tp)
    for s in range(0, len(p), chunk):
        above = p[None, s:s + chunk] > GRID[:, None, None]
        tp += (above & (y[None, s:s + chunk] > 0)).sum(1); npred += above.sum(1)
    return GRID[(2 * tp / (npred + y.sum(0)[None] + 1e-9)).argmax(0)]

half = np.random.default_rng(1).permutation(len(Y)) < len(Y) // 2

def honest(p):
    """Thresholds tuned on one half of the training set (out-of-fold probabilities), F1 measured on the other half."""
    return wf1(Y[~half], decide(p[~half], best_thresholds(p[half], Y[half])))

## 1. Image: frozen SigLIP embeddings (computed once, `out/siglip_feats.npz`), then a multi-label head

In [ ]:
f_img = OUT / "siglip_feats.npz"
if "image" in MODELS and not f_img.exists():
    import open_clip
    from PIL import Image
    from torch.utils.data import DataLoader, Dataset
    IMG = pathlib.Path("/content/images")     # images copied to Colab's local disk (Drive is slow)
    if not IMG.exists():
        !cp "{D}/images_NFkV58V.tar" /content/ && tar -xf /content/images_NFkV58V.tar -C /content/ && rm /content/images_NFkV58V.tar
    model, _, preprocess = open_clip.create_model_and_transforms("ViT-SO400M-14-SigLIP", "webli")
    tokenizer = open_clip.get_tokenizer("ViT-SO400M-14-SigLIP")
    model = model.to(dev).eval()

    class Imgs(Dataset):
        def __init__(self, names): self.names = list(names)
        def __len__(self): return len(self.names)
        def __getitem__(self, i):
            try:
                return preprocess(Image.open(IMG / self.names[i]).convert("RGB"))
            except Exception:          # missing or unreadable image: black image
                return torch.zeros(3, 224, 224)

    def encode(names):
        out = []
        dl = DataLoader(Imgs(names), batch_size=256, num_workers=os.cpu_count())
        with torch.no_grad(), torch.autocast("cuda", dtype=amp):
            for x in dl:
                out.append(torch.nn.functional.normalize(model.encode_image(x.to(dev)), dim=-1).float().cpu().numpy())
        return np.concatenate(out).astype(np.float16)

    # zero-shot scores: similarity between the image and "a photo of a red product", etc.
    words = {"Multiple Colors": "multicolored"}
    prompts = [f"a photo of a {words.get(c, c.lower())} product" for c in COLORS]
    with torch.no_grad(), torch.autocast("cuda", dtype=amp):
        T = torch.nn.functional.normalize(model.encode_text(tokenizer(prompts).to(dev)), dim=-1).float().cpu().numpy()
    np.savez(f_img, train=encode(Xtr.image_file_name), test=encode(Xte.image_file_name), text=T)
    del model; torch.cuda.empty_cache()

In [ ]:
import torch.nn as nn

def train_head(x, y, xs, epochs=25, seed=0):
    torch.manual_seed(seed)
    net = nn.Sequential(nn.Dropout(0.2), nn.Linear(x.shape[1], 1024), nn.GELU(), nn.Dropout(0.4),
                        nn.Linear(1024, y.shape[1])).to(dev)
    opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-2)
    steps = epochs * ((len(x) + 511) // 512)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=1e-3, total_steps=steps)
    x, y = torch.tensor(x).to(dev), torch.tensor(y).to(dev)
    for _ in range(epochs):
        net.train()
        for idx in torch.randperm(len(x), device=dev).split(512):
            loss = nn.functional.binary_cross_entropy_with_logits(net(x[idx]), y[idx])
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    net.eval()
    with torch.no_grad():
        return [torch.sigmoid(net(torch.tensor(a).to(dev))).cpu().numpy() for a in xs]

if "image" in MODELS and todo("image"):
    z = np.load(f_img)
    Ftr, Fte, T = z["train"].astype(np.float32), z["test"].astype(np.float32), z["text"]
    Atr = np.hstack([Ftr * 10, Ftr @ T.T * 10]); Ate = np.hstack([Fte * 10, Fte @ T.T * 10])
    for k in todo("image"):
        a, b = FOLDS[k]
        pv, pt = train_head(Atr[a], Y[a], [Atr[b], Ate], seed=k)
        np.savez(fold_file("image", k), val=pv, test=pt)
        print(f"image fold {k}: F1 at threshold 0.5 = {wf1(Y[b], pv > 0.5):.4f}", flush=True)
    del Ftr, Fte, Atr, Ate; gc.collect()
print("image: missing folds", todo("image"))

## 2. Text: character TF-IDF + one logistic regression per colour (CPU)

In [ ]:
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

if "tfidf" in MODELS and todo("tfidf"):
    titles = [clean(s) for s in pd.concat([Xtr.item_name, Xte.item_name])]
    captions = [clean(s)[:1000] for s in pd.concat([Xtr.item_caption, Xte.item_caption])]
    v_title = TfidfVectorizer(analyzer="char", ngram_range=(1, 4), min_df=3, max_features=500_000, sublinear_tf=True)
    v_caption = TfidfVectorizer(analyzer="char", ngram_range=(2, 3), min_df=5, max_features=300_000, sublinear_tf=True)
    A = sp.hstack([v_title.fit_transform(titles), 0.5 * v_caption.fit_transform(captions)]).tocsr()
    Atr, Ate = A[:len(Xtr)], A[len(Xtr):]
    for k in todo("tfidf"):
        a, b = FOLDS[k]
        clf = OneVsRestClassifier(LogisticRegression(C=4, solver="liblinear"), n_jobs=-1).fit(Atr[a], Y[a].astype(int))
        pv, pt = clf.predict_proba(Atr[b]), clf.predict_proba(Ate)
        np.savez(fold_file("tfidf", k), val=pv, test=pt)
        print(f"tfidf fold {k}: F1 at threshold 0.5 = {wf1(Y[b], pv > 0.5):.4f}", flush=True)
    del A, Atr, Ate, clf; gc.collect()
print("tfidf: missing folds", todo("tfidf"))

## 3. Text: Japanese BERT fine-tuned for multi-label classification, one fold after another (≈ 23 min per fold on A100)

In [ ]:
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments,
                          DataCollatorWithPadding, set_seed)
from datasets import Dataset as HFDataset

if "bert" in MODELS and todo(BERT_NAME):
    assert dev == "cuda", "BERT needs a GPU"
    tok = AutoTokenizer.from_pretrained(TEXT_MODEL)
    def ds(texts, labels=None):
        d = {"text": list(texts)}
        if labels is not None: d["labels"] = [list(r) for r in labels]
        return HFDataset.from_dict(d).map(lambda b: tok(b["text"], truncation=True, max_length=TEXT_MAX_LEN),
                                          batched=True, remove_columns=["text"])
    ds_te = ds(Xte.text.values)
    for k in todo(BERT_NAME):
        a, b = FOLDS[k]
        set_seed(SEED + k)
        model = AutoModelForSequenceClassification.from_pretrained(
            TEXT_MODEL, num_labels=len(COLORS), problem_type="multi_label_classification")
        args = TrainingArguments(
            output_dir="/content/txt", num_train_epochs=TEXT_EPOCHS, learning_rate=TEXT_LR,
            per_device_train_batch_size=TEXT_BS, per_device_eval_batch_size=256, warmup_ratio=0.06,
            weight_decay=0.01, bf16=amp == torch.bfloat16, fp16=amp == torch.float16,
            group_by_length=True, logging_steps=200, save_strategy="no", report_to="none", seed=SEED + k)
        trainer = Trainer(model=model, args=args, train_dataset=ds(Xtr.text.values[a], Y[a]), tokenizer=tok,
                          data_collator=DataCollatorWithPadding(tok))
        trainer.train()
        trainer.args.group_by_length = False          # otherwise the prediction order is shuffled
        sig = lambda d: torch.sigmoid(torch.tensor(trainer.predict(d).predictions).float()).numpy()
        pv, pt = sig(ds(Xtr.text.values[b])), sig(ds_te)
        np.savez(fold_file(BERT_NAME, k), val=pv, test=pt)
        print(f"{BERT_NAME} fold {k}: F1 at threshold 0.5 = {wf1(Y[b], pv > 0.5):.4f}", flush=True)
        del model, trainer; gc.collect(); torch.cuda.empty_cache()
print(BERT_NAME, ": missing folds", todo(BERT_NAME))

## 4. Stacking on the 212,120 out-of-fold rows

Only the models whose **5 folds** are all available enter the stacking.

In [ ]:
from sklearn.linear_model import LogisticRegression

names = [n for n in ("image", "tfidf", BERT_NAME) if merge(n) is not None]
P = {n: merge(n) for n in names}
print("complete models:", names)
for n, (o, _) in P.items():
    print(f"  {n:15s} alone: {honest(o):.4f}")

logit = lambda p: np.log(np.clip(p, 1e-5, 1 - 1e-5) / (1 - np.clip(p, 1e-5, 1 - 1e-5)))
S_tr = np.hstack([logit(o) for o, _ in P.values()])
S_te = np.hstack([logit(t) for _, t in P.values()])
Yi = Y.astype(int)
META = list(KFold(5, shuffle=True, random_state=7).split(S_tr))   # meta-model folds, different from FOLDS

def stacker(x, y):
    return LogisticRegression(C=0.5, max_iter=2000).fit(x, y)

lr_oof = np.zeros(Y.shape)
for a, b in META:
    for j in range(len(COLORS)):
        lr_oof[b, j] = stacker(S_tr[a], Yi[a, j]).predict_proba(S_tr[b])[:, 1]
lr_te = np.column_stack([stacker(S_tr, Yi[:, j]).predict_proba(S_te)[:, 1] for j in range(len(COLORS))])
s_lr = honest(lr_oof)
print(f"  stacking (logistic regression): {s_lr:.4f}")

## 4b. Catalogue neighbours (same shop, close item numbers) + one XGBoost per colour

Image file names look like `<shop>_<item>_1.jpg`, and a shop lists its products in batches. For each product: the
colours of the 5 previous and 5 next training products of the same shop (the product itself excluded), and the shop's
usual colours computed **out-of-fold** (without the product's fold, so its own colour does not leak).
This reflects how the catalogue was assembled, not the product itself: report it as such.

In [ ]:
# image_file_name = <shop>_<item>_1.jpg
def ids(X):
    p = X.image_file_name.str.extract(r"^(\d+)_(\d+)")
    shop, item = p[0].astype(np.int64).values, p[1].astype(np.int64).values
    return shop, shop * 10**9 + item
shop_tr, key_tr = ids(Xtr); shop_te, key_te = ids(Xte)
order = np.argsort(key_tr, kind="stable")
s_key, s_shop, s_Y = key_tr[order], shop_tr[order], Y[order]
rank = np.empty(len(order), dtype=np.int64); rank[order] = np.arange(len(order))
K = 5

def neigh(qkey, qshop, self_rank=None):
    """Mean colours of the K training neighbours on each side (same shop), the product itself excluded."""
    pos = np.searchsorted(s_key, qkey)
    acc, w = np.zeros((len(qkey), len(COLORS)), np.float32), np.zeros(len(qkey), np.float32)
    near = np.full(len(qkey), 99.0, np.float32)
    for side in (-1, 1):
        for j in range(1, K + 1):
            if self_rank is not None:
                idx = self_rank + side * j
            else:
                idx = pos - j if side < 0 else pos + j - 1
            ok = (idx >= 0) & (idx < len(s_key))
            idx = idx.clip(0, len(s_key) - 1)
            ok &= s_shop[idx] == qshop
            acc += ok[:, None] * s_Y[idx]; w += ok
            if j == 1:
                near = np.minimum(near, np.where(ok, np.log1p(np.abs(s_key[idx] - qkey)), 99.0))
    return np.hstack([acc / np.maximum(w, 1)[:, None], w[:, None], near[:, None]])

# The shop's usual colours, out-of-fold: for a product of fold k, computed without fold k
# (removing only the product itself would leak its colour)
M, prior = 5.0, Y.mean(0)
def shop_stats(rows):
    g = pd.DataFrame(Y[rows], columns=COLORS).assign(shop=shop_tr[rows]).groupby("shop")
    return g[COLORS].sum(), g.size()
def shop_prior(shop, sums, cnt):
    sm = sums.reindex(shop).fillna(0).values; n = cnt.reindex(shop).fillna(0).values
    return np.hstack([(sm + M * prior) / (n[:, None] + M), np.log1p(n)[:, None]]).astype(np.float32)
V_tr = np.zeros((len(Xtr), len(COLORS) + 1), np.float32)
for a, b in FOLDS:
    V_tr[b] = shop_prior(shop_tr[b], *shop_stats(a))
V_te = shop_prior(shop_te, *shop_stats(np.arange(len(Xtr))))

F_tr = np.hstack([S_tr, neigh(key_tr, shop_tr, rank), V_tr]).astype(np.float32)
F_te = np.hstack([S_te, neigh(key_te, shop_te), V_te]).astype(np.float32)
c_w = S_tr.shape[1] + len(COLORS)
print("features:", F_tr.shape, " at least one neighbour: train", (F_tr[:, c_w] > 0).mean().round(3),
      "test", (F_te[:, c_w] > 0).mean().round(3))

In [ ]:
# One XGBoost per colour (on GPU: a few seconds per model)
import xgboost as xgb
params = dict(n_estimators=400, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.5,
              min_child_weight=5, reg_lambda=1.0, tree_method="hist", device=dev, verbosity=0)

xgb_oof = np.zeros(Y.shape)
for i, (a, b) in enumerate(META):
    for j in range(len(COLORS)):
        xgb_oof[b, j] = xgb.XGBClassifier(**params).fit(F_tr[a], Yi[a, j]).predict_proba(F_tr[b])[:, 1]
    print("meta fold", i, "done", flush=True)
xgb_te = np.column_stack([xgb.XGBClassifier(**params).fit(F_tr, Yi[:, j]).predict_proba(F_te)[:, 1]
                          for j in range(len(COLORS))])
s_xgb = honest(xgb_oof)
print(f"stacking + neighbours (XGBoost): {s_xgb:.4f}   |   stacking without neighbours: {s_lr:.4f}")

## 5. Submission: the better of the two stackers (per-colour thresholds tuned on the 212,120 out-of-fold rows)

In [ ]:
from datetime import date
from google.colab import files

choices = {"neighbours": (s_xgb, xgb_oof, xgb_te), "stack": (s_lr, lr_oof, lr_te)}
fusion, (score, p_oof, p_te) = max(choices.items(), key=lambda c: c[1][0])
print("selected:", fusion)
th = best_thresholds(p_oof, Y)
pred = decide(p_te, th)
tags = [str([c for c, k in zip(COLORS, row) if k]) for row in pred]   # e.g. "['Black', 'White']"
sub = pd.DataFrame({"color_tags": tags}, index=Xte.index)
fname = f"colours5_{fusion}_{'-'.join(names)}_cv{score:.4f}_{date.today()}.csv"
sub.to_csv(OUT / fname)
np.savez(OUT / fname.replace(".csv", "_probs.npz"), test=p_te, th=th)
print(dict(zip(COLORS, th.round(2))))
print(sub.head(), "\n", fname, len(sub), "rows, colours per product:", pred.sum(1).mean().round(2))
files.download(str(OUT / fname))